In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder\
        .master("local")\
        .appName("Colab")\
        .config('spark.ui.port', '4050')\
        .getOrCreate()
spark


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 22:11:58 WARN Utils: Your hostname, dev, resolves to a loopback address: 127.0.1.1; using 192.168.0.3 instead (on interface wlo1)
26/06/09 22:11:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 22:11:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 22:12:00 WARN Utils: Service 'SparkUI' could not bind on port 4050. Attempting port 4051.


# Import Libraries

In [2]:
from pyspark.sql.functions import col, lit, count, abs, first, round, sum
from pyspark.sql.types import StringType, IntegerType, DoubleType

## Loading Data

In [3]:
path = "../data/raw/"

In [ ]:
geo = spark.read.parquet(path + "geo/", header=True)
print(f"Number of Rows = {geo.count()}")
geo.show(5)

26/06/09 22:12:27 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/geo/*.
java.io.FileNotFoundException: File ../data/raw/geo/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

Number of Rows = 611959
+------------------+-----------------+--------------------+--------------------+
|           latitud|         longitud|              comuna|                  ID|
+------------------+-----------------+--------------------+--------------------+
|  353894.669721904|6301392.600521904|b13b671cb296c1ce5...|ab6f6062e7fac953a...|
| 297273.7487498676|6271440.347349868|d10ad8071d7270bc1...|b12fac130e9008be6...|
|172956.24831402366|5702581.978114024|20c5891e1d78fe2f3...|cf29a4e836a1c4ba4...|
| 135846.1650072791|5920764.953007279|c51ed7a673a2184f2...|46b48d3aa7694ae78...|
|351678.75214929937|6282760.015749299|8e7e23148e55a25a0...|c86eb4ca0aeb0a981...|
+------------------+-----------------+--------------------+--------------------+
only showing top 5 rows


In [8]:
labels = spark.read.parquet("drive/My Drive/DE_Test/data_sample/labels/*", header=True)
print(f"Number of Rows = {labels.count()}")
labels.show(5)

Number of Rows = 84435
+--------------------+-----+
|                  ID|event|
+--------------------+-----+
|ea165b785d74859a9...|    2|
|558d0ed3e3cf87a3d...|    2|
|2da14424526a7d741...|    2|
|8f619aaa096c4e9da...|    2|
|cabf40eac3538e1a6...|    1|
+--------------------+-----+
only showing top 5 rows



# Step 1: Data Profiling & Cleaning
Before any transformation, inspect both tables and document your findings.
Add a comment block at the end of this section summarizing what you found and what you did about it.

## 1a. Schema validation and row counts

In [ ]:
# Validate schema and row counts for both tables
print('=== geo ===')
print(f'Rows: {geo.count()}')
geo.printSchema()

print('=== labels ===')
print(f'Rows: {labels.count()}')
labels.printSchema()


## 1b. NULL detection in key fields

In [ ]:
from pyspark.sql.functions import col, count, when, isnan

# Check NULLs in geo: ID, comuna, latitud, longitud
geo.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ['ID', 'comuna', 'latitud', 'longitud']
]).show()

# Check NULLs in labels: ID, event
labels.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ['ID', 'event']
]).show()

# TODO: decide how to handle rows with NULL IDs — drop or flag?


## 1c. Coordinate range validation

In [ ]:
# Inspect coordinate ranges — are they WGS84 (lat -90/+90, lon -180/+180)
# or a projected system (e.g. UTM in meters)?
geo.select(
    col('latitud').cast('double'),
    col('longitud').cast('double')
).summary('min', 'max', 'mean', 'stddev').show()

# TODO: document the coordinate system you identified and whether
# any values fall outside the expected range for this dataset.


## 1d. Duplicate detection and deduplication strategy

In [ ]:
from pyspark.sql.functions import count, col

# Count duplicate IDs in geo
geo_dupes = geo.groupBy('ID').count().filter(col('count') > 1)
print(f'Duplicate IDs in geo: {geo_dupes.count()}')

# Count duplicate IDs in labels
lab_dupes = labels.groupBy('ID').count().filter(col('count') > 1)
print(f'Duplicate IDs in labels: {lab_dupes.count()}')

# Count ID overlap between tables
overlap = geo.select('ID').distinct().join(labels.select('ID').distinct(), 'ID')
print(f'IDs present in both tables: {overlap.count()}')

# ── DEDUPLICATION STRATEGY (required) ────────────────────────────────────
# Before writing code, document your strategy here:
#
# Strategy chosen (e.g. first, most frequent, most recent):
#   <your answer>
#
# Why 'first' may or may not be appropriate for this dataset:
#   <your answer>
# ─────────────────────────────────────────────────────────────────────────

# TODO: implement deduplication for geo and labels


## 1e. Profiling summary
Add a comment block below documenting: total rows kept/dropped per table, NULLs handled, coordinate system identified, and deduplication decision.

In [ ]:
# PROFILING SUMMARY
# -----------------
# geo table:
#   Original rows      :
#   Rows after cleaning:
#   NULLs dropped      :
#   Duplicates removed :
#   Coordinate system  :
#
# labels table:
#   Original rows      :
#   Rows after cleaning:
#   NULLs dropped      :
#   Duplicates removed :
#
# ID overlap (geo ∩ labels):
# Impact on downstream join :


# Step 2: Join & Transform
Apply the cleaning decisions from Step 1, then join and transform the data.

In [9]:
# Step 1 result: apply your deduplication from the profiling section
# geo_clean should have unique IDs with the first value of comuna, latitud, longitud
# Use the strategy you documented in cell 16

geo_clean = None  # TODO: replace with your deduplication implementation

# Uncomment to verify:
# print(f'geo_clean rows: {geo_clean.count()}')
# geo_clean.show(5)


+--------------------+--------------------+------------------+-----------------+
|                  ID|              comuna|           latitud|         longitud|
+--------------------+--------------------+------------------+-----------------+
|00ea32329fa4525da...|8e7e23148e55a25a0...|355003.62541433144|6282015.927125556|
|0210f55e81f73f3da...|4b20bf8e091932655...| 274527.8736461363|6682088.686543643|
|036d964becbfe696d...|b13b671cb296c1ce5...| 351527.3253498232|6301151.171749824|
|03b2afdcd1da9be16...|8e7e23148e55a25a0...|352943.88487017597|6277854.775532687|
|051cef5baa043ddf3...|b13b671cb296c1ce5...| 353076.7622881545|6300903.916888154|
+--------------------+--------------------+------------------+-----------------+
only showing top 5 rows



In [10]:
# Step 2: join clean geo with clean labels through the ID field
# Use the join type you justified in your profiling summary
# Then apply the transformations needed to prepare the distance calculation

temp = None  # TODO: replace with your join and transformation implementation

# Expected output columns: comuna, latitud_2, longitud_2, event counts
# Uncomment to verify:
# print(f'temp rows: {temp.count()}')
# temp.show(5)


18776
+--------------------+---------+----------+----+---+
|              comuna|latitud_2|longitud_2|   1|  2|
+--------------------+---------+----------+----+---+
|8e7e23148e55a25a0...| 355089.8| 6282631.9|null|  1|
|8e7e23148e55a25a0...| 352735.1| 6283233.8|null|  1|
|8e7e23148e55a25a0...| 354686.4| 6283647.4|null|  1|
|8e7e23148e55a25a0...| 354746.6| 6284267.6|null|  1|
|5b14d89ceb61cab2c...| 346185.4| 6305164.3|null|  1|
+--------------------+---------+----------+----+---+
only showing top 5 rows



# Step 3: Distance Calculation
Identify, for each customer, which labeled events occur within a 50-meter radius.
Before implementing, document your optimization strategy in the cell below.

In [ ]:
# Before implementing the distance calculation, estimate the scale of the problem:
# How many potential customer-event pairs exist? What does that mean for memory and compute?
# Use this analysis to justify your optimization strategy in the next cell.

# TODO: compute the scale estimate here


Customers x Events = 611959 x 84435 = 51670758165


In [ ]:
# ── OPTIMIZATION STRATEGY (required) ────────────────────────────────────────
# Before writing code, document your strategy in the block below (5-10 lines):
#
# Chosen approach:
#   <describe the strategy you selected, e.g. spatial bucketing, BallTree index,
#    broadcast join, grid partitioning, etc.>
#
# Why this approach fits this dataset:
#   <justify based on data characteristics: volume, coordinate system, commune
#    distribution, skew, memory constraints, etc.>
#
# Trade-off vs. at least one alternative you considered:
#   <e.g. 'A cross-join would be simpler but produces O(n*m) combinations;
#    my approach reduces this to O(k) per spatial bucket because...'>
# ─────────────────────────────────────────────────────────────────────────────

## Implement your distance mapping below.
## Input:  'temp' DataFrame (columns: comuna, latitud_2, longitud_2, event counts)
## Output: 'mapping' DataFrame with one row per customer pair within 50m,
##          including the calculated distance and event types of each pair.

mapping = None  # replace with your implementation

# Uncomment when ready:
# print(f"Number of Rows in dataframe 'mapping' = {mapping.count()}")
# mapping.show(5)


Number of Rows in dataframe 'mapping' = 18776


In [ ]:
## For each customer, calculate the average distance to all labeled events
## within the 50m radius, aggregated by event type.
## Expected output columns: customer_id (or spatial key), event_type, avg_distance, count_events

# Your implementation here


In [ ]:
# Step 4: Define Output Table Schemas
# The output schema may differ depending on the consumer. Define both below.

# ── Schema A: for Data Scientists (rich feature table) ───────────────────
# Fields: <list columns, types, and purpose>
# Format: <e.g. Parquet, Delta Lake>
# Partitioning: <e.g. by date, by comuna>
# Access: <e.g. read directly from S3 into notebook>
#
# ── Schema B: for Business Analysts (SQL-queryable) ──────────────────────
# Fields: <list columns, types, and purpose — may be aggregated/simplified>
# Format: <e.g. table in Athena, Redshift, Snowflake>
# Partitioning: <e.g. by comuna, by event_type>
# Access: <e.g. exposed via SQL view>
#
# Justify any differences between Schema A and Schema B.


In [ ]:
# Step 5: Save Output
# Save the result of the distance calculation with support for weekly history.
# The file/table must allow querying a specific week without reprocessing all data.
#
# Chosen format and path structure:
#   <e.g. s3://bucket/abt_result/year=2025/week=23/part-*.parquet>
#
# Why this structure supports weekly history efficiently:
#   <your justification>

# TODO: implement the save logic here
# mapping.write.mode('overwrite') \
#     .partitionBy('year', 'week') \
#     .parquet('<your output path>')


# Point 3: SQL & Data Quality
Connect the notebook to SQLite, load the required tables, and complete the exercises below.
For each query, add a brief comment explaining: (1) the business question it answers and (2) what data quality issue it could reveal.

In [ ]:
# Connection sqlite and init Database
import sqlite3

conn = sqlite3.connect('datalake_stage.db')
print("Opened database successfully");

curs = conn.cursor()

In [ ]:
# Load required tables into SQLite
# Use the cleaned DataFrames from Step 1 and the result from Step 3

# Example: load clean geo table
# df_geo_clean = geo_clean.toPandas()
# df_geo_clean.to_sql(con=conn, name='tbl_geo_clean', if_exists='replace', index=False)

# TODO: load the labels table
# df_labels = labels.toPandas()
# df_labels.to_sql(con=conn, name='tbl_labels', if_exists='replace', index=False)

# TODO: load your result table from Step 3 (distance mapping output)
# df_result = mapping.toPandas()
# df_result.to_sql(con=conn, name='tbl_abt_result', if_exists='replace', index=False)

print('Tables loaded.')


In [ ]:
# Close connection to save the db in file datalake_stage.db
conn.close()

In [ ]:
# We will first load an sql extension into our environment
# This extension will allow us to work with sql on Colaboratory
#
%load_ext sql

# We will then connect to our in memory sqlite database
# NB: This database will cease to exist as soon as the database connection is closed.
# We will learn more about how databases are created later in prep.
#
%sql sqlite:///datalake_stage.db
# Nothing to do here

In [ ]:
%%sql
# Example query on cell magic
select *
from tbl_land_geo_orig
where comuna ='b13b671cb296c1ce5eb94117f308118364cd258b322f61872cc7364dfcf5f2ad'
limit 10

In [ ]:
%%sql
-- Exercise 1: Data Quality Checks
-- (a) Count rows with NULL values in any key field (ID, comuna, event)
-- (b) Detect duplicate IDs after deduplication
-- (c) Find customers whose coordinates fall outside the expected range for their
--     commune. Use a subquery to compute the bounding box (min/max lat & lon)
--     per commune, then flag customers that fall outside it.
--
-- Your queries here:


In [ ]:
%%sql
-- Exercise 2: Window Function — Top 20 communes by type 2 events
-- Use RANK() or DENSE_RANK() over the total count of type 2 events per commune.
-- Include a second column with each commune's share (%) of the national total
-- for type 2 events. Use a CTE or subquery to compute the national total.
--
-- Expected columns: comuna, total_type2_events, national_rank, pct_of_national_total
--
-- Your query here:


In [ ]:
%%sql
-- Exercise 3: CTE — Type 1 events by commune with geographic outlier detection
-- Using a CTE, calculate per commune: avg, max and min of latitude and longitude,
-- event count, and the distance between the max and min latitude points.
-- In the outer query, return only communes where the avg latitude deviates
-- more than 10% from the overall national average.
--
-- Expected columns: comuna, avg_lat, max_lat, min_lat, avg_lon, max_lon, min_lon,
--                    event_count, lat_range, deviation_pct
--
-- Your query here:


In [ ]:
%%sql
-- Exercise 4: Conditional aggregation — event type comparison per commune
-- In a single query (no JOINs), use conditional aggregation to show per commune:
-- count of type 1 events, count of type 2 events, and the ratio type_2 / type_1.
-- Order by ratio descending.
-- Handle communes where type_1 = 0 to avoid division by zero.
--
-- Expected columns: comuna, count_type1, count_type2, ratio_type2_type1
--
-- Your query here:
